# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track"
REPO_DIR = "FlyRank-ML-Track"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ML-Notebooks/FlyRank-ML-Track/FlyRank-ML-Track
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I chose Refresh/Content Opportunity Scoring mainly because it matches my existing experience with machine learning best — I've worked with classification, scoring, and evaluation-style problems before, and this lane's shape feels closest to that. It also just feels like the best fit for what I've seen of the dataset so far — the columns line up with a "which page needs review" question in a way I find intuitive.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df.columns)

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The question I'm working to answer is "Out of pages with real traffic, which 50 should a reviewer look at first this week, and what should they do with each?" The unit of analysis is a single page. The decision this improves is where a reviewer with limited weekly capacity spends their attention, replacing a hand-tuned rule with a ranked, reasoned queue.

A wrong call costs differently by direction: flagging a page that wasn't a priority wastes capacity, while missing a real risk means a page keeps declining, or an at-risk page goes unprotected. ML helps because what separates "review now" from "fine as-is" is several signals combining, not one threshold — which is reflected in the way that earlier examples in the first two notebooks suggested a large difference between the effectiveness of the hand-rule and a proper model.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# "Real traffic" — pages actually worth reviewing at all (Section 2's scope)
real_traffic = (df["impressions_90d"] > 0) & (df["clicks_90d"] > 0)
df_active = df[real_traffic].copy()

print(f"Total pages in starter dataset: {len(df)}")
print(f"Pages with real traffic (90d): {len(df_active)} ({len(df_active)/len(df):.1%})")

# Build the decline proxy directly from trend_direction (Section 4's caveat: proxy, not verified outcome)
declining = df_active["trend_direction"] == "down"
print(f"Share of active pages currently trending down: {declining.mean():.1%}")

# Do declining pages look meaningfully different on a couple of review-relevant signals?
# (freshness and position are the two review-action signals Section 2 leans on)
print("\nAvg days_since_last_update:")
print(f"  Trending down: {df_active.loc[declining, 'days_since_last_update'].mean():.1f}")
print(f"  Not trending down: {df_active.loc[~declining, 'days_since_last_update'].mean():.1f}")

print("\nAvg position_tier value counts (trending down pages):")
print(df_active.loc[declining, "position_tier"].value_counts(normalize=True).round(3))

Total pages in starter dataset: 30000
Pages with real traffic (90d): 16796 (56.0%)
Share of active pages currently trending down: 57.8%

Avg days_since_last_update:
  Trending down: 51.6
  Not trending down: 49.1

Avg position_tier value counts (trending down pages):
position_tier
page_1      0.451
striking    0.270
page_3_5    0.239
top_3       0.035
deep        0.006
Name: proportion, dtype: float64


Based on what is observed from running the numbers above on the starter dataset 56.0% of pages have real traffic (impressions and clicks in the trailing 90 days), those being the pool this lane actually applies to. Of those, 57.8% are currently trending down, which is a large enough share that a reviewer clearly can't eyeball all of them, reinforcing the capacity problem Section 2 is built around.

The freshness gap is smaller than I expected, with trending-down pages averaging 51.6 days since last update versus 49.1 for stable pages — only a ~2.5 day difference, which suggests staleness alone doesn't explain most of the decline here.

The position-tier breakdown adds more nuance as trending-down pages are concentrated in page_1 (45.1%) and striking distance (27.0%), meaning a large share of declining pages are actually ranking well or near-well right now — they're not simply low-visibility pages fading further, but often visible pages already at risk of losing position.

Together, these numbers suggest decline here isn't explained by any single signal (freshness, position) on its own, which is itself part of the argument for a model over a hand-written rule: the useful pattern likely comes from how these signals combine, not from any one of them alone.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My work will be able to say, for a given page, whether it currently shows measured signs of decline or a demand/click mismatch, derived from facts about this window rather than forecasts — including the decline label itself, which is a proxy signal rather than a verified outcome. It'll also provide the reason why it was flagged and a suggested action. That lets strategists know what pages to prioritize for review and potentially overhaul. However, it will not guarantee that reviewing or refreshing a flagged page will cause recovery, as that can't be determined from this alone.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.